<a href="https://colab.research.google.com/github/aadhikesavanr/F1_ML_Project/blob/main/F1_ML_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas numpy
!pip install scikit-learn requests joblib


In [ ]:
import requests
import pandas as pd

all_rows = []

for year in range(2000, 2026):
    print(f"Downloading {year}...")

    url = f"https://ergast.com/api/f1/{year}/results.json?limit=2000"

    try:
        data = requests.get(url).json()
        races = data["MRData"]["RaceTable"]["Races"]

        for race in races:
            for result in race["Results"]:
                all_rows.append({
                    "season": year,
                    "race": race["raceName"],
                    "circuit": race["Circuit"]["circuitName"],
                    "driver": result["Driver"]["familyName"],
                    "constructor": result["Constructor"]["name"],
                    "grid": result["grid"],
                    "position": result["position"]
                })

    except Exception as e:
        print(f"Error in {year}: {e}")

df = pd.DataFrame(all_rows)

print("Dataset Shape:", df.shape)
df.head()


Error in 2000: Expecting value: line 1 column 1 (char 0)
Error in 2001: Expecting value: line 1 column 1 (char 0)
Error in 2002: Expecting value: line 1 column 1 (char 0)
Error in 2003: Expecting value: line 1 column 1 (char 0)
Error in 2004: Expecting value: line 1 column 1 (char 0)
Error in 2005: Expecting value: line 1 column 1 (char 0)
Error in 2006: Expecting value: line 1 column 1 (char 0)
Error in 2007: Expecting value: line 1 column 1 (char 0)
Error in 2008: Expecting value: line 1 column 1 (char 0)
Error in 2009: Expecting value: line 1 column 1 (char 0)
Error in 2010: Expecting value: line 1 column 1 (char 0)
Error in 2011: Expecting value: line 1 column 1 (char 0)
Error in 2012: Expecting value: line 1 column 1 (char 0)
Error in 2013: Expecting value: line 1 column 1 (char 0)
Error in 2014: Expecting value: line 1 column 1 (char 0)
Error in 2015: Expecting value: line 1 column 1 (char 0)
Error in 2016: Expecting value: line 1 column 1 (char 0)
Error in 2017: Expecting value:

""


In [ ]:
pip install requests



SyntaxError: invalid syntax (1619862723.py, line 1)

In [ ]:
pip install requests


In [ ]:
import requests
import csv
import time

BASE_URL = "https://api.jolpi.ca/ergast/f1"

START_SEASON = 2015
END_SEASON = 2025

OUTPUT_FILE = "f1_race_results_2015_2025.csv"

HEADERS = {
    "User-Agent": "F1ResultsDownloader/1.0"
}

FIELDS = [
    "season",
    "round",
    "race_name",
    "race_date",
    "circuit",
    "country",
    "driver_id",
    "driver_number",
    "driver_code",
    "driver_first_name",
    "driver_last_name",
    "driver_nationality",
    "constructor_id",
    "constructor_name",
    "constructor_nationality",
    "grid",
    "position",
    "position_text",
    "points",
    "laps",
    "status",
    "fastest_lap_rank",
    "fastest_lap",
    "fastest_lap_time",
    "average_speed"
]


def get_season_results(season):
    """
    Download all race results for a season.

    Jolpica has a maximum limit of 100 results per request,
    so pagination is used.
    """

    all_races = []
    offset = 0
    limit = 100

    while True:

        url = f"{BASE_URL}/{season}/results/"

        params = {
            "limit": limit,
            "offset": offset
        }

        response = requests.get(
            url,
            params=params,
            headers=HEADERS,
            timeout=30
        )

        # Show useful information if request failed
        if response.status_code != 200:
            print(f"    HTTP {response.status_code}")
            print(f"    URL: {response.url}")
            print(f"    Response: {response.text[:300]}")
            response.raise_for_status()

        # Make sure we actually received JSON
        try:
            data = response.json()
        except ValueError:
            print("    API did not return valid JSON")
            print(f"    Response: {response.text[:300]}")
            raise

        mrdata = data["MRData"]

        races = mrdata["RaceTable"]["Races"]

        all_races.extend(races)

        total = int(mrdata["total"])

        offset += limit

        if offset >= total:
            break

    return all_races


def flatten_results(races):

    rows = []

    for race in races:

        season = race.get("season")
        round_number = race.get("round")
        race_name = race.get("raceName")
        race_date = race.get("date")

        circuit = race.get("Circuit", {})

        circuit_name = circuit.get("circuitName")

        location = circuit.get("Location", {})

        country = location.get("country")

        results = race.get("Results", [])

        for result in results:

            driver = result.get("Driver", {})
            constructor = result.get("Constructor", {})

            fastest_lap = result.get("FastestLap", {})

            fastest_lap_time = (
                fastest_lap.get("Time", {})
                .get("time")
            )

            average_speed = (
                fastest_lap.get("AverageSpeed", {})
                .get("speed")
            )

            row = {
                "season": season,
                "round": round_number,
                "race_name": race_name,
                "race_date": race_date,
                "circuit": circuit_name,
                "country": country,

                "driver_id": driver.get("driverId"),
                "driver_number": result.get("number"),
                "driver_code": driver.get("code"),
                "driver_first_name": driver.get("givenName"),
                "driver_last_name": driver.get("familyName"),
                "driver_nationality": driver.get("nationality"),

                "constructor_id": constructor.get("constructorId"),
                "constructor_name": constructor.get("name"),
                "constructor_nationality": constructor.get("nationality"),

                "grid": result.get("grid"),
                "position": result.get("position"),
                "position_text": result.get("positionText"),
                "points": result.get("points"),
                "laps": result.get("laps"),
                "status": result.get("status"),

                "fastest_lap_rank": fastest_lap.get("rank"),
                "fastest_lap": fastest_lap.get("lap"),
                "fastest_lap_time": fastest_lap_time,
                "average_speed": average_speed
            }

            rows.append(row)

    return rows


def main():

    all_results = []

    for season in range(START_SEASON, END_SEASON + 1):

        print(f"Downloading {season}...")

        try:

            races = get_season_results(season)

            rows = flatten_results(races)

            all_results.extend(rows)

            print(
                f"  {len(races)} races, "
                f"{len(rows)} driver results"
            )

        except Exception as e:

            print(f"  ERROR in {season}: {e}")

        # Don't hammer the API
        time.sleep(1)

    print()
    print("Saving CSV...")

    with open(
        OUTPUT_FILE,
        "w",
        newline="",
        encoding="utf-8"
    ) as file:

        writer = csv.DictWriter(
            file,
            fieldnames=FIELDS
        )

        writer.writeheader()
        writer.writerows(all_results)

    print()
    print("Done!")
    print(f"Total rows: {len(all_results)}")
    print(f"File: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()


  22 races, 378 driver results
  25 races, 462 driver results
  20 races, 400 driver results
  21 races, 420 driver results
  21 races, 420 driver results
  17 races, 340 driver results
  22 races, 440 driver results
  22 races, 440 driver results
  22 races, 440 driver results
  28 races, 479 driver results
  27 races, 479 driver results

Saving CSV...

Done!
Total rows: 4698
File: f1_race_results_2015_2025.csv


In [1]:
print(f"Total rows:{len(all_results)}")

NameError: name 'all_results' is not defined

In [2]:
from google.colab import files

files.download("f1_race_results_2015_2025.csv")

FileNotFoundError: Cannot find file: f1_race_results_2015_2025.csv

In [3]:
import requests
import csv
import time

BASE_URL = "https://api.jolpi.ca/ergast/f1"

START_SEASON = 2015
END_SEASON = 2025

OUTPUT_FILE = "f1_race_results_2015_2025.csv"

HEADERS = {
    "User-Agent": "F1ResultsDownloader/1.0"
}

FIELDS = [
    "season",
    "round",
    "race_name",
    "race_date",
    "circuit",
    "country",
    "driver_id",
    "driver_number",
    "driver_code",
    "driver_first_name",
    "driver_last_name",
    "driver_nationality",
    "constructor_id",
    "constructor_name",
    "constructor_nationality",
    "grid",
    "position",
    "position_text",
    "points",
    "laps",
    "status",
    "fastest_lap_rank",
    "fastest_lap",
    "fastest_lap_time",
    "average_speed"
]


def get_season_results(season):
    """
    Download all race results for a season.

    Jolpica has a maximum limit of 100 results per request,
    so pagination is used.
    """

    all_races = []
    offset = 0
    limit = 100

    while True:

        url = f"{BASE_URL}/{season}/results/"

        params = {
            "limit": limit,
            "offset": offset
        }

        response = requests.get(
            url,
            params=params,
            headers=HEADERS,
            timeout=30
        )

        # Show useful information if request failed
        if response.status_code != 200:
            print(f"    HTTP {response.status_code}")
            print(f"    URL: {response.url}")
            print(f"    Response: {response.text[:300]}")
            response.raise_for_status()

        # Make sure we actually received JSON
        try:
            data = response.json()
        except ValueError:
            print("    API did not return valid JSON")
            print(f"    Response: {response.text[:300]}")
            raise

        mrdata = data["MRData"]

        races = mrdata["RaceTable"]["Races"]

        all_races.extend(races)

        total = int(mrdata["total"])

        offset += limit

        if offset >= total:
            break

    return all_races


def flatten_results(races):

    rows = []

    for race in races:

        season = race.get("season")
        round_number = race.get("round")
        race_name = race.get("raceName")
        race_date = race.get("date")

        circuit = race.get("Circuit", {})

        circuit_name = circuit.get("circuitName")

        location = circuit.get("Location", {})

        country = location.get("country")

        results = race.get("Results", [])

        for result in results:

            driver = result.get("Driver", {})
            constructor = result.get("Constructor", {})

            fastest_lap = result.get("FastestLap", {})

            fastest_lap_time = (
                fastest_lap.get("Time", {})
                .get("time")
            )

            average_speed = (
                fastest_lap.get("AverageSpeed", {})
                .get("speed")
            )

            row = {
                "season": season,
                "round": round_number,
                "race_name": race_name,
                "race_date": race_date,
                "circuit": circuit_name,
                "country": country,

                "driver_id": driver.get("driverId"),
                "driver_number": result.get("number"),
                "driver_code": driver.get("code"),
                "driver_first_name": driver.get("givenName"),
                "driver_last_name": driver.get("familyName"),
                "driver_nationality": driver.get("nationality"),

                "constructor_id": constructor.get("constructorId"),
                "constructor_name": constructor.get("name"),
                "constructor_nationality": constructor.get("nationality"),

                "grid": result.get("grid"),
                "position": result.get("position"),
                "position_text": result.get("positionText"),
                "points": result.get("points"),
                "laps": result.get("laps"),
                "status": result.get("status"),

                "fastest_lap_rank": fastest_lap.get("rank"),
                "fastest_lap": fastest_lap.get("lap"),
                "fastest_lap_time": fastest_lap_time,
                "average_speed": average_speed
            }

            rows.append(row)

    return rows


def main():

    all_results = []

    for season in range(START_SEASON, END_SEASON + 1):

        print(f"Downloading {season}...")

        try:

            races = get_season_results(season)

            rows = flatten_results(races)

            all_results.extend(rows)

            print(
                f"  {len(races)} races, "
                f"{len(rows)} driver results"
            )

        except Exception as e:

            print(f"  ERROR in {season}: {e}")

        # Don't hammer the API
        time.sleep(1)

    print()
    print("Saving CSV...")

    with open(
        OUTPUT_FILE,
        "w",
        newline="",
        encoding="utf-8"
    ) as file:

        writer = csv.DictWriter(
            file,
            fieldnames=FIELDS
        )

        writer.writeheader()
        writer.writerows(all_results)

    print()
    print("Done!")
    print(f"Total rows: {len(all_results)}")
    print(f"File: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()


  22 races, 378 driver results
  25 races, 462 driver results
  20 races, 400 driver results
  21 races, 420 driver results
  21 races, 420 driver results
  17 races, 340 driver results
  22 races, 440 driver results
  22 races, 440 driver results
  22 races, 440 driver results
  28 races, 479 driver results
  27 races, 479 driver results

Saving CSV...

Done!
Total rows: 4698
File: f1_race_results_2015_2025.csv


In [4]:
from google.colab import files

files.download("f1_race_results_2015_2025.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>